In [ ]:
import os
import subprocess
import sys

os.environ["PYTHONUNBUFFERED"] = "1"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
try:
    subprocess.check_call(["nvidia-smi"])
except FileNotFoundError:
    print("nvidia-smi not found", flush=True)
probe = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    text=True,
).strip()
print(probe, flush=True)
if not probe.endswith("True"):
    raise RuntimeError("CUDA is not available")


# Second-life garment routing — front_only_384

Main training only. Overfit sanity lives in `kaggle_overfit.ipynb` / `configs/usage_overfit.yaml` and must not block this run.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ["PYTHONUNBUFFERED"] = "1"

def find_one(root: Path, name: str) -> Path:
    matches = [path for path in root.rglob(name) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError((name, matches))
    return matches[0]

input_root = Path("/kaggle/input")
pyprojects = [
    path for path in input_root.rglob("pyproject.toml")
    if (path.parent / "src" / "garmentroute").is_dir()
]
if len(pyprojects) == 1:
    package_root = pyprojects[0].parent
else:
    package_root = Path("/kaggle/working/package")
    package_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(find_one(input_root, "package.zip")) as archive:
        archive.extractall(package_root)

work_package = Path("/kaggle/working/package")
if package_root.resolve() != work_package.resolve():
    if work_package.exists():
        shutil.rmtree(work_package)
    shutil.copytree(package_root, work_package)
    package_root = work_package

subprocess.check_call(
    [sys.executable, "-u", "-m", "pip", "install", "--no-deps", "--no-build-isolation", str(package_root)]
)
print("package", package_root, flush=True)


In [ ]:
import zipfile
from pathlib import Path

import pandas as pd

def find_one(root: Path, name: str) -> Path:
    matches = [path for path in root.rglob(name) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError((name, matches))
    return matches[0]

def image_layout(root: Path) -> tuple[Path, Path]:
    index_path = root / "index.parquet"
    if (root / "resized").is_dir():
        return index_path, root / "resized"
    if (root / "station1").is_dir():
        return index_path, root
    raise FileNotFoundError(root)

input_root = Path("/kaggle/input")
# Prefer an explicit bundle extract so we never train on a half-ready dataset mount.
bundle_matches = [path for path in input_root.rglob("bundle.zip") if path.is_file()]
data_root = Path("/kaggle/working/data")
if bundle_matches:
    if data_root.exists():
        import shutil
        shutil.rmtree(data_root)
    data_root.mkdir(parents=True, exist_ok=True)
    print("extracting", bundle_matches[0], flush=True)
    with zipfile.ZipFile(bundle_matches[0]) as archive:
        archive.extractall(data_root)
else:
    parquets = [path for path in input_root.rglob("index.parquet") if path.is_file()]
    if len(parquets) != 1:
        raise FileNotFoundError(("index.parquet", parquets))
    data_root = parquets[0].parent

index_path, image_root = image_layout(data_root)
frame = pd.read_parquet(index_path)
n_jpeg = sum(1 for _ in image_root.rglob("*.jpg"))
print(data_root, flush=True)
print(frame["split"].value_counts().to_string(), flush=True)
print("jpeg files", n_jpeg, flush=True)
# 10k draw needs ~32k member images; old 6k mount had 11800.
if n_jpeg < 30000:
    raise RuntimeError(f"image dataset not ready for 10k train: jpeg={n_jpeg} (need >=30000)")


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from garmentroute.train import train
from garmentroute.util import load_yaml

WORKING = Path("/kaggle/working")

def stamp(label: str) -> None:
    print(label, datetime.now(timezone.utc).isoformat(), flush=True)

def apply_paths(config: dict) -> dict:
    config["index"] = str(index_path)
    config["image_root"] = str(image_root)
    config["run_dir"] = str(WORKING / "runs")
    rel = config.get("subset_members")
    if rel:
        candidate = Path(rel)
        if not candidate.is_file():
            name = Path(rel).name
            for root in (package_root, package_root / "data", index_path.parent, index_path.parent / "data"):
                for path in (root / rel, root / name):
                    if path.is_file():
                        candidate = path
                        break
                else:
                    continue
                break
        if candidate.is_file():
            config["subset_members"] = str(candidate)
            print("subset_members", candidate, flush=True)
        else:
            print("WARNING: subset_members not found:", rel, flush=True)
    return config

overfit_cfg = package_root / "configs" / "usage_overfit.yaml"
if not overfit_cfg.is_file():
    print(
        "WARNING: usage_overfit.yaml missing; run notebooks/kaggle_overfit.ipynb separately if needed.",
        flush=True,
    )
else:
    print(
        "NOTE: overfit sanity is configs/usage_overfit.yaml + notebooks/kaggle_overfit.ipynb; "
        "this notebook only trains front_only_384.",
        flush=True,
    )

front = apply_paths(load_yaml(package_root / "configs" / "front_only_384.yaml"))
stamp("START front_only_384.yaml")
front_checkpoint = train(front, package_root / "configs" / "front_only_384.yaml")
stamp("END front_only_384.yaml")
print("checkpoint", front_checkpoint, flush=True)
